In [1]:
import pandas as pd

df = pd.read_csv("talabalar_baholari 777.csv", parse_dates=["sana"])
df.head()

,talaba_id,ism,guruh,fan,imtihon,sana,ball
0,1,Talaba 001,DA-101,Matematika,Oraliq 1,2025-10-15,56.0
1,1,Talaba 001,DA-101,Matematika,Oraliq 2,2025-11-20,61.0
2,1,Talaba 001,DA-101,Matematika,Yakuniy,2025-12-25,72.0
3,1,Talaba 001,DA-101,Python,Oraliq 1,2025-10-15,60.0
4,1,Talaba 001,DA-101,Python,Oraliq 2,2025-11-20,72.0


In [2]:
df["kelmagan"] = df["ball"].isna()

guruh_hisobot = df.groupby("guruh").agg(
    talabalar=("talaba_id", "nunique"),
    qatorlar=("ball", "size"),
    baholar=("ball", "count"),
    kelmaganlar=("kelmagan", "sum"),
    ortacha_ball=("ball", "mean")
).round(2)

guruh_hisobot

,talabalar,qatorlar,baholar,kelmaganlar,ortacha_ball
guruh,,,,,
DA-101,28,420,399,21,70.15
DA-102,25,375,362,13,67.20
DA-103,30,450,426,24,72.32
DA-104,12,144,135,9,66.55


In [3]:
print(guruh_hisobot["talabalar"].sum() == df["talaba_id"].nunique())
print(guruh_hisobot["qatorlar"].sum() == len(df))

True
True


In [4]:
guruh_fan = df.pivot_table(
    index="guruh",
    columns="fan",
    values="ball",
    aggfunc="mean"
).round(2)

guruh_fan

fan,Ingliz tili,Matematika,Python,SQL,Statistika
guruh,,,,,
DA-101,71.60,66.14,73.97,71.99,67.15
DA-102,69.25,62.53,71.80,69.03,63.38
DA-103,75.58,65.57,76.90,74.81,68.99
DA-104,67.50,61.37,73.03,NaN,64.50


In [5]:
df[(df["guruh"] == "DA-104") & (df["fan"] == "SQL")]

,talaba_id,ism,guruh,fan,imtihon,sana,ball,kelmagan


In [6]:
talaba_ort = df.groupby(["guruh", "talaba_id"])["ball"].mean()
alochilar = (talaba_ort > 75).groupby("guruh").mean().mul(100).round(1)
alochilar

,ball
guruh,
DA-101,28.6
DA-102,12.0
DA-103,43.3
DA-104,0.0


In [7]:
print(len(talaba_ort) == df["talaba_id"].nunique())
print((talaba_ort > 75).groupby("guruh").sum())

True
guruh
DA-101     8
DA-102     3
DA-103    13
DA-104     0
Name: ball, dtype: int64


In [8]:
df["daraja"] = pd.cut(
    df["ball"],
    bins=[-1, 55, 70, 85, 100],
    labels=["2", "3", "4", "5"]
)

daraja_foiz = pd.crosstab(
    df["guruh"],
    df["daraja"],
    normalize="index"
).mul(100).round(1)

daraja_foiz

daraja,2,3,4,5
guruh,,,,
DA-101,13.0,39.1,35.8,12.0
DA-102,14.9,45.9,32.6,6.6
DA-103,10.8,31.7,41.3,16.2
DA-104,12.6,52.6,32.6,2.2


In [9]:
print(daraja_foiz.sum(axis=1))
print(pd.crosstab(df["guruh"], df["daraja"]).values.sum() == df["ball"].notna().sum())

guruh
DA-101     99.9
DA-102    100.0
DA-103    100.0
DA-104    100.0
dtype: float64
True


In [10]:
import numpy as np

df["guruh_ort"] = df.groupby(["guruh", "fan", "imtihon"])["ball"].transform("mean").round(2)
df["farq"] = (df["ball"] - df["guruh_ort"]).round(2)
df["holat"] = np.select(
    [df["farq"] > 0, df["farq"] < 0, df["farq"] == 0],
    ["Yuqori", "Past", "Teng"],
    default="Kelmagan"
)

df[["talaba_id", "guruh", "fan", "imtihon", "ball", "guruh_ort", "farq", "holat"]].head(10)

,talaba_id,guruh,fan,imtihon,ball,guruh_ort,farq,holat
0,1,DA-101,Matematika,Oraliq 1,56.0,66.71,-10.71,Past
1,1,DA-101,Matematika,Oraliq 2,61.0,67.96,-6.96,Past
2,1,DA-101,Matematika,Yakuniy,72.0,63.62,8.38,Yuqori
3,1,DA-101,Python,Oraliq 1,60.0,71.96,-11.96,Past
4,1,DA-101,Python,Oraliq 2,72.0,76.50,-4.50,Past
5,1,DA-101,Python,Yakuniy,77.0,73.50,3.50,Yuqori
6,1,DA-101,Statistika,Oraliq 1,67.0,66.42,0.58,Yuqori
7,1,DA-101,Statistika,Oraliq 2,55.0,70.81,-15.81,Past
8,1,DA-101,Statistika,Yakuniy,62.0,64.43,-2.43,Past
9,1,DA-101,Ingliz tili,Oraliq 1,NaN,71.96,NaN,Kelmagan


In [11]:
print(df["holat"].value_counts())
print("\nUzunlik tengmi:", len(df["farq"]) == len(df))

holat
Yuqori      661
Past        659
Kelmagan     67
Teng          2
Name: count, dtype: int64

Uzunlik tengmi: True


In [12]:
fan_imtihon = df.pivot_table(
    index="fan",
    columns="imtihon",
    values="ball",
    aggfunc="mean"
)[["Oraliq 1", "Oraliq 2", "Yakuniy"]].round(2)

fan_imtihon

imtihon,Oraliq 1,Oraliq 2,Yakuniy
fan,,,
Ingliz tili,73.16,70.94,71.21
Matematika,63.68,65.59,63.89
Python,73.64,75.01,73.87
SQL,71.05,72.24,72.95
Statistika,66.08,67.99,65.19


In [13]:
print("NaN lar soni:", fan_imtihon.isna().sum().sum())

NaN lar soni: 0


In [14]:
kelganlar = df.groupby("talaba_id").filter(lambda g: g["ball"].notna().all())

soni = kelganlar["talaba_id"].nunique()
ulush = round(soni / df["talaba_id"].nunique() * 100, 1)

print(f"Hamma imtihonga kelganlar: {soni} nafar ({ulush}%)")

Hamma imtihonga kelganlar: 46 nafar (48.4%)


In [15]:
print(kelganlar["ball"].isna().sum() == 0)

True


In [16]:
# Yakuniy imtihonlarda har talaba bo'yicha ballning o'rtachasi
yakuniy = df[df["imtihon"] == "Yakuniy"]

top3 = (
    yakuniy.groupby(["talaba_id", "ism", "guruh"])["ball"]
    .mean()
    .nlargest(3)
    .round(2)
)

top3

,,,ball
talaba_id,ism,guruh,
28,Talaba 028,DA-101,94.0
61,Talaba 061,DA-103,91.2
70,Talaba 070,DA-103,86.2


In [17]:
tekshiruv = yakuniy.groupby(["talaba_id", "ism", "guruh"])["ball"].agg(
    topshirgan="count",
    jami="size",
    ortacha="mean"
).round(2)

tekshiruv.loc[top3.index]

,,,topshirgan,jami,ortacha
talaba_id,ism,guruh,,,
28,Talaba 028,DA-101,3,5,94.0
61,Talaba 061,DA-103,5,5,91.2
70,Talaba 070,DA-103,5,5,86.2
